In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import requests
import os
from google.colab import drive

# Drive mount
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

input_file = 'Mediafire_Direct_Links_Updated.xlsx'
save_path = '/content/drive/MyDrive/mediafire_novels_backup'

# Browser-like Session
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
    'Connection': 'keep-alive'
})

if os.path.exists(input_file):
    df = pd.read_excel(input_file)
    for index, row in df.iterrows():
        d_link = row['Direct Download Links']
        if pd.notna(d_link) and str(d_link).startswith('http'):
            file_name = d_link.split('/')[-1].split('?')[0]
            full_save_path = os.path.join(save_path, file_name)

            # Check if file is already a "fake" small file (verification page)
            if os.path.exists(full_save_path) and os.path.getsize(full_save_path) > 50000: # 50KB se bari file sahi hogi
                print(f"Skip {index+1}: Already exists.")
                continue

            try:
                print(f"Attempting {index+1}: {file_name}")
                r = session.get(d_link, stream=True, timeout=30)

                # Check agar page HTML ha (yani block ho gaya ha) ya asli PDF
                if 'text/html' in r.headers.get('Content-Type', ''):
                    print(f"!!! Error {index+1}: Mediafire ne block kar diya (Captcha Required).")
                    break # Stop and try later

                with open(full_save_path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=1024*1024):
                        f.write(chunk)
                print(f"Success {index+1}")
                time.sleep(2) # Zyada delay taake block na hon

            except Exception as e:
                print(f"Error {index+1}: {e}")

Attempting 1: Bandhe+ek+dor+se+by+alisha+naz.pdf
!!! Error 1: Mediafire ne block kar diya (Captcha Required).


In [3]:
import pandas as pd
import requests
import os
from google.colab import drive

# 1. Google Drive mount karen (Agar pehle nahi kiya)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. File Check
input_file = 'Mediafire_Direct_Links_Updated.xlsx'

if not os.path.exists(input_file):
    print(f"Error: {input_file} nahi mili! Left side par 'Files' icon par click karke file upload karen.")
else:
    df = pd.read_excel(input_file)

    # 3. Drive ma folder setup
    save_path = '/content/drive/MyDrive/mediafire_novels_backup'
    if not os.path.exists(save_path):
        os.makedirs(save_path)
        print(f"Folder ban gaya: {save_path}")

    print(f"Transfer shuru ho raha ha... Total links: {len(df)}")

    # 4. Loop with Resume Logic
    for index, row in df.iterrows():
        d_link = row['Direct Download Links']

        if pd.notna(d_link) and str(d_link).startswith('http'):
            try:
                # File name nikalna
                file_name = d_link.split('/')[-1].split('?')[0]
                # File name ko thora saaf karna agar zaroorat ho
                full_save_path = os.path.join(save_path, file_name)

                # --- RESUME LOGIC ---
                # Check agar file pehle se Drive ma mojud ha
                if os.path.exists(full_save_path):
                    print(f"Skip {index+1}: {file_name} pehle se mojud ha.")
                    continue

                print(f"Transferring {index+1}/{len(df)}: {file_name}")
                with requests.get(d_link, stream=True, timeout=30) as r:
                    r.raise_for_status()
                    with open(full_save_path, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=1024*1024): # 1MB chunks
                            if chunk:
                                f.write(chunk)

            except Exception as e:
                print(f"Error index {index+1}: {e}")
        else:
            print(f"Skip index {index+1}: Link missing ha.")

    print("\n--- Tamam files check aur transfer ho chuki hain! ---")

Folder ban gaya: /content/drive/MyDrive/mediafire_novels_backup
Transfer shuru ho raha ha... Total links: 4
Transferring 1/4: Bandhe+ek+dor+se+by+alisha+naz.pdf
Transferring 2/4: Man+Raksam+By+Saba+Baloch+.pdf
Transferring 3/4: Dastaan+Hidayat+O+Roshni+Ki+By+Tahreem+Fatima+%28Complete+Novel%29.pdf
Transferring 4/4: Ae+dill+By+umaima+mukarram.pdf
Error index 4: 502 Server Error: Bad Gateway for url: http://www.mediafire.com/download_repair.php?flag=2&dkey=mye5twzcfmfgqSjeEWHvhcvbuYt9ZGc_XMAn9GUgmkDxBCApWoyFSVkEycDTWiFFZacykHTsT0Pzl8XDig3uQKAEZU-U7euL3LKIfFr7IjlcQFUkMDIm7T1v78kNu_t9apR30BmWU6x0cGJuWopBWY3cyBZbANGo4hhf19fXi3LPFfc&qkey=qhn7is5knf67su1&ip=34.73.54.8

--- Tamam files check aur transfer ho chuki hain! ---
